
# Seed comparison: synthetic vs LLM vs ESConv (full grid)

Runs full grids over all emotions and strategies (5 trials each) for each seeding mode:

- Synthetic templates
- LLM-generated seeds
- ESConv conversation starts (classifier-checked first-turn openers)

Each section saves CSV/heatmap/meta/log under `results/single_turn_[seed]_grid/`.


In [ ]:
!pip uninstall -y dynamic-conversation dynamic_conversation
!pip cache purge
!pip install --no-cache-dir git+https://github.com/Javin-Mendiratta/Dynamic-Conversation.git@derek_12_13


from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OpenAI')


from pathlib import Path
import json
from datasets import load_dataset
from dynamic_conversation import (
    SingleTurnSimulator,
    SimulationConfig,
    ResponseStrategy,
    build_esconv_seed_bank,
)

results_root = Path("results")
results_root.mkdir(exist_ok=True)

cfg = SimulationConfig(
    model="gpt-4o-mini",
    max_tokens=800,
    temperature=1.0,
    brevity_hint="Reply in 1-2 sentences, keep emotion visible."
)

emotions = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
strategies = list(ResponseStrategy)
style_mod = "concise and emotionally attuned"

# Build ESConv seed bank (first-turn openers, classifier-checked)
esconv = load_dataset("thu-coai/esconv")
seed_bank = build_esconv_seed_bank(esconv, emotions=emotions, per_emotion=8, max_chars=200, use_gpu=False)


In [ ]:

def run_and_report(seed_label, use_llm_seed=False, use_esconv_seed=False, esconv_seeds=None):
    out_dir = results_root / f"single_turn_{seed_label}_grid"
    out_dir.mkdir(parents=True, exist_ok=True)
    sim = SingleTurnSimulator(
        use_gpu=False,
        config=cfg,
        prompt_for_key=True,
        esconv_seeds=esconv_seeds,
    )
    df = sim.run_batch(
        emotions=emotions,
        strategies=strategies,
        runs_per_pair=5,
        style_modifier=style_mod,
        use_llm_seed=use_llm_seed,
        include_baseline=True,
        use_esconv_seed=use_esconv_seed,
        save_csv=out_dir / f"single_turn_{seed_label}.csv",
        save_heatmap=out_dir / f"single_turn_{seed_label}_heatmap.png",
    )
    meta_path = out_dir / f"single_turn_{seed_label}.meta.json"
    log_path = out_dir / f"single_turn_{seed_label}.log"
    meta = json.load(open(meta_path)) if meta_path.exists() else {}
    log = log_path.read_text() if log_path.exists() else ""
    return df, meta, log, out_dir


## Synthetic seeds

In [ ]:

print("Running synthetic seeding...")
df_syn, meta_syn, log_syn, dir_syn = run_and_report("synthetic", use_llm_seed=False, use_esconv_seed=False)
print("Saved to", dir_syn)
print("Meta:", meta_syn)
print("Log:", log_syn)

df_syn.head()


## LLM seeds

In [ ]:

print("Running LLM seeding...")
df_llm, meta_llm, log_llm, dir_llm = run_and_report("llm", use_llm_seed=True, use_esconv_seed=False)
print("Saved to", dir_llm)
print("Meta:", meta_llm)
print("Log:", log_llm)

df_llm.head()

## ESConv seeds

In [ ]:

print("Running ESConv seeding...")
df_esconv, meta_esconv, log_esconv, dir_esconv = run_and_report("esconv", use_llm_seed=False, use_esconv_seed=True, esconv_seeds=seed_bank)
print("Saved to", dir_esconv)
print("Meta:", meta_esconv)
print("Log:", log_esconv)

df_esconv.head()
